<a href="https://colab.research.google.com/github/cbonnin88/mes_allocs-ETL/blob/main/amplitude_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import requests
import json
import gdown as gd
import polars as pl
from google.colab import userdata
from tqdm import tqdm

In [16]:
url = 'https://drive.google.com/uc?id=1-miKL2ruAqnj7UxeaE84YO_UCsY8v4pS'
gd.download(url,'mes_allocs_events.csv',quiet=True)

'mes_allocs_events.csv'

In [17]:
df_events = pl.read_csv('mes_allocs_events.csv')

In [18]:
df_events.head()

user_id,event_type,timestamp,device_id,device_type,event_properties,user_properties
str,str,i64,str,str,str,str
"""user_2199""","""benefit_selected""",1767225600000,"""dev_9479""","""Mobile""","""{""page_url"": ""https://www.mes-…","""{""app_version"": ""2.4.0"", ""env""…"
"""user_9407""","""page_view""",1767225604000,"""dev_4894""","""Desktop""","""{""page_url"": ""https://www.mes-…","""{""app_version"": ""2.4.0"", ""env""…"
"""user_6914""","""application_submitted""",1767225608000,"""dev_8929""","""Desktop""","""{""page_url"": ""https://www.mes-…","""{""app_version"": ""2.4.0"", ""env""…"
"""user_13475""","""page_view""",1767225629000,"""dev_6467""","""Mobile""","""{""page_url"": ""https://www.mes-…","""{""app_version"": ""2.4.0"", ""env""…"
"""user_7470""","""benefit_selected""",1767225663000,"""dev_8305""","""Desktop""","""{""page_url"": ""https://www.mes-…","""{""app_version"": ""2.4.0"", ""env""…"


# **Configuration**

In [23]:
amplitude_key = userdata.get('mes-allocs-key')
batch_url = 'https://api.eu.amplitude.com/batch'
batch_size = 1000

# **Preparing the Data**

In [24]:
all_events = df_events.to_dicts()

print(f'Starting upload of {len(all_events)} events')

Starting upload of 600000 events


# **Iterative Batch Upload**

In [26]:
for i in tqdm(range(0,len(all_events), batch_size)):
  batch_slice = all_events[i : i + batch_size]

  # Format each event for Amplitude Schema
  payload_events = []
  for e in batch_slice:
    payload_events.append({
        'user_id': str(e['user_id']),
        'event_type': e['event_type'],
        'time': int(e['timestamp']),
        'device_id': str(e['timestamp']),
        'app_version': '2.4.0',
        'platform': e['device_type'],
        'event_properties': json.loads(e['event_properties']),
        'user_properties': json.loads(e['user_properties'])
    })

  # Sending the request
  response = requests.post(
      batch_url,
      headers={'Content-Type':'application/json'},
      json={
          'api_key':amplitude_key,
          'events':payload_events
      }
  )

  if response.status_code != 200:
    print(f'❌ Error at batch {i}: {response.text}')
    break

print('✅ Upload Complete! Head over to Amplitude to see your real-time charts.')


100%|██████████| 600/600 [11:56<00:00,  1.19s/it]

✅ Upload Complete! Head over to Amplitude to see your real-time charts.
